# 02 - CWGAN (baseline arm)

Trains the conditional WGAN with weight clipping and writes its synthetic pool
(`N_MAX` rows per class). GPU. Run once per dataset. See `PIPELINE.md`, step 2.

In [ ]:
# Colab: upload df_train_<dataset>.csv (and ckpt_*.pt to resume an interrupted run).
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    files.upload()

In [ ]:
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import OneHotEncoder

try:
    display
except NameError:
    display = print

## Config

In [ ]:
DATASET  = "diabetes"     # diabetes | yeast | ecoli | pageblocks
DATA_DIR = "."

SETTING = dict(updates=10000, batch=250, lr=2e-4, noise_dim=128)   # shared by every arm

SEED        = 42
CLIP_VALUE  = 0.01        # weight clipping range [-c, c]
CRITIC_ITER = 5           # critic steps per generator update
MAX_DISCRETE_LEVELS = 12  # columns with at most this many levels are snapped back

MAX_MINUTES    = 150      # stop and checkpoint before a Colab session is cut
RESUME         = True     # continue from ckpt_*.pt when present
CHECKPOINT_DIR = "."      # a Google Drive path survives a disconnect
SAVE_EVERY     = 1000     # generator updates between checkpoints

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. In Colab: Runtime > Change runtime type > T4 GPU.")
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, f"df_train_{DATASET}.csv"))
TARGET_COL = df_train.columns[-1]
FEATURES = [c for c in df_train.columns if c != TARGET_COL]

_v = df_train[FEATURES].to_numpy(dtype=float)
assert _v.min() >= -1e-6 and _v.max() <= 1 + 1e-6, "features are not scaled to [0, 1]"

COUNTS  = df_train[TARGET_COL].value_counts().sort_index()
CLASSES = COUNTS.index.to_numpy()
N_MAX   = int(COUNTS.max())

input_dim = len(FEATURES)
cond_dim  = len(CLASSES)
ohe = OneHotEncoder(sparse_output=False, categories=[CLASSES])
ohe.fit(CLASSES.reshape(-1, 1))

print("train:", df_train.shape, "| class counts:", COUNTS.tolist())
print(f"pool: {N_MAX:,} rows x {cond_dim} classes")

## Model

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim + cond_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, input_dim), nn.Sigmoid(),
        )

    def forward(self, z, y):
        return self.net(torch.cat([z, y], dim=1))


class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim + cond_dim, 256), nn.LeakyReLU(0.2), nn.Dropout(0.5),
            nn.Linear(256, 256), nn.LeakyReLU(0.2), nn.Dropout(0.5),
            nn.Linear(256, 1),
        )

    def forward(self, x, y):
        return self.net(torch.cat([x, y], dim=1))

## Discrete-value schema (post-hoc snapping)

In [ ]:
def build_schema(df_reference, target_col):
    schema = {"discrete": {}, "continuous": {}}
    for col in [c for c in df_reference.columns if c != target_col]:
        levels = np.unique(df_reference[col].dropna().to_numpy(dtype=float))
        if len(levels) <= MAX_DISCRETE_LEVELS:
            schema["discrete"][col] = np.sort(levels)
        else:
            schema["continuous"][col] = (float(df_reference[col].min()),
                                         float(df_reference[col].max()))
    return schema


def apply_schema(df_synth, schema):
    """Snap discrete columns to legal levels, clip continuous ones to range."""
    out = df_synth.copy()
    for col, levels in schema["discrete"].items():
        v = out[col].to_numpy(dtype=float)
        out[col] = levels[np.abs(v[:, None] - levels[None, :]).argmin(axis=1)]
    for col, (lo, hi) in schema["continuous"].items():
        out[col] = out[col].clip(lo, hi)
    return out


def validity(df, schema):
    """Share of discrete cells that hold a legal level."""
    ok = [np.isin(df[c].to_numpy(dtype=float), lv).mean() for c, lv in schema["discrete"].items()]
    return 100.0 * float(np.mean(ok)) if ok else 100.0


SCHEMA = build_schema(df_train, TARGET_COL)
print("discrete columns:", len(SCHEMA["discrete"]), "| continuous:", len(SCHEMA["continuous"]))

## Checkpointing

In [ ]:
def mount_if_drive(path):
    if str(path).startswith("/content/drive"):
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
        os.makedirs(path, exist_ok=True)
    return path


CHECKPOINT_DIR = mount_if_drive(CHECKPOINT_DIR)


def trial_key(c):
    return f"u{c['updates']}_b{c['batch']}_lr{c['lr']:.0e}_z{c['noise_dim']}"


def ckpt_path(stem):
    return os.path.join(CHECKPOINT_DIR, f"ckpt_{stem}.pt")


def rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def rng_restore(s):
    random.setstate(s["python"])
    np.random.set_state(s["numpy"])
    torch.set_rng_state(s["torch"].cpu() if hasattr(s["torch"], "cpu") else s["torch"])
    if s.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([t.cpu() for t in s["cuda"]])


def save_ckpt(stem, cfg, G, D, oG, oD, done, history):
    payload = {"version": 3, "kind": "cwgan", "dataset": DATASET, "cfg": cfg,
               "updates_done": done, "history": history,
               "G": G.state_dict(), "D": D.state_dict(),
               "opt_G": oG.state_dict(), "opt_D": oD.state_dict(),
               "rng": rng_state(), "input_dim": input_dim, "cond_dim": cond_dim,
               "classes": CLASSES.tolist(), "seed": SEED}
    tmp = ckpt_path(stem) + ".tmp"
    torch.save(payload, tmp)
    os.replace(tmp, ckpt_path(stem))


def load_ckpt(stem, cfg):
    p = ckpt_path(stem)
    if not (RESUME and os.path.isfile(p)):
        return None
    c = torch.load(p, map_location=DEVICE, weights_only=False)
    if c.get("kind") != "cwgan" or c.get("dataset") != DATASET:
        raise ValueError(f"{p} belongs to a different notebook or dataset")
    if trial_key(c["cfg"]) != trial_key(cfg):
        raise ValueError(f"{p} was trained with {trial_key(c['cfg'])}, not {trial_key(cfg)}")
    return c

## Training

In [ ]:
def train_cwgan(df, cfg, stem, max_minutes, seed=SEED):
    """Train for cfg['updates'] generator updates (each with CRITIC_ITER critic steps).
    Stops and checkpoints when max_minutes runs out; call again to resume."""
    X_t = torch.tensor(df[FEATURES].to_numpy(dtype="float32"), device=DEVICE)
    y_t = torch.tensor(ohe.transform(df[[TARGET_COL]].to_numpy()).astype("float32"),
                       device=DEVICE)
    batch = int(cfg["batch"])
    log_every = max(1, cfg["updates"] // 200)

    G = Generator(cfg["noise_dim"]).to(DEVICE)
    D = Critic().to(DEVICE)
    oG = torch.optim.Adam(G.parameters(), lr=cfg["lr"], betas=(0.5, 0.9))
    oD = torch.optim.Adam(D.parameters(), lr=cfg["lr"], betas=(0.5, 0.9))

    ck = load_ckpt(stem, cfg)
    if ck is not None:
        G.load_state_dict(ck["G"]); D.load_state_dict(ck["D"])
        oG.load_state_dict(ck["opt_G"]); oD.load_state_dict(ck["opt_D"])
        rng_restore(ck["rng"])
        done, history = ck["updates_done"], list(ck["history"])
        print(f"resumed at {done:,} / {cfg['updates']:,} updates")
    else:
        set_seed(seed)
        done, history = 0, []
    if done >= cfg["updates"]:
        return G, history, done

    deadline = time.time() + max_minutes * 60
    t_start, started_at = time.time(), done
    for u in range(done, cfg["updates"]):
        for _ in range(CRITIC_ITER):
            idx = np.random.randint(0, len(X_t), batch)
            real_x, real_y = X_t[idx], y_t[idx]
            z = torch.randn(batch, cfg["noise_dim"], device=DEVICE)
            fake_x = G(z, real_y)
            loss_D = -(D(real_x, real_y).mean() - D(fake_x.detach(), real_y).mean())
            oD.zero_grad(set_to_none=True); loss_D.backward(); oD.step()
            with torch.no_grad():                       # Lipschitz by weight clipping
                for p in D.parameters():
                    p.clamp_(-CLIP_VALUE, CLIP_VALUE)

        z = torch.randn(batch, cfg["noise_dim"], device=DEVICE)
        loss_G = -D(G(z, real_y), real_y).mean()
        oG.zero_grad(set_to_none=True); loss_G.backward(); oG.step()

        done = u + 1
        if done % log_every == 0 or done == cfg["updates"]:
            history.append((done, float(loss_D.detach()), float(loss_G.detach())))
        if done % SAVE_EVERY == 0:
            save_ckpt(stem, cfg, G, D, oG, oD, done, history)
        if done % (log_every * 10) == 0:
            rate = (done - started_at) / max(time.time() - t_start, 1e-9)
            print(f"  update {done:>7,} / {cfg['updates']:,} | D {float(loss_D):9.4f} "
                  f"| G {float(loss_G):9.4f} | {rate:6.1f} upd/s")
        if time.time() > deadline:
            print(f"time limit at {done:,} updates - re-run to resume from the checkpoint")
            break

    save_ckpt(stem, cfg, G, D, oG, oD, done, history)
    return G, history, done


def generate_pool(G, cfg, per_class, chunk=131072):
    """per_class synthetic rows for every class, majority included."""
    G.eval()
    parts = []
    with torch.no_grad():
        for cls in CLASSES:
            cond = ohe.transform(np.array([[cls]])).astype("float32")
            made = 0
            while made < per_class:
                n = min(chunk, per_class - made)
                z = torch.randn(n, cfg["noise_dim"], device=DEVICE)
                y = torch.tensor(np.repeat(cond, n, axis=0), device=DEVICE)
                part = pd.DataFrame(G(z, y).cpu().numpy(), columns=FEATURES)
                part[TARGET_COL] = cls
                parts.append(part)
                made += n
    G.train()
    return pd.concat(parts, ignore_index=True)[df_train.columns]

## Run

In [ ]:
STEM = f"cwgan_{DATASET}_{trial_key(SETTING)}_seed{SEED}"
steps_per_epoch = max(1, len(df_train) // SETTING["batch"])
print("setting   :", trial_key(SETTING))
print("equivalent:", f"{SETTING['updates'] / steps_per_epoch:.1f} epochs on this dataset")

G, history, done = train_cwgan(df_train, SETTING, STEM, MAX_MINUTES)
COMPLETE = done >= SETTING["updates"]
print("training complete:", COMPLETE)

In [ ]:
if COMPLETE:
    df_pool = generate_pool(G, SETTING, N_MAX)
    print(f"valid discrete cells before snapping: {validity(df_pool, SCHEMA):.2f}%")
    df_pool = apply_schema(df_pool, SCHEMA)
    print(f"valid discrete cells after snapping : {validity(df_pool, SCHEMA):.2f}%")

    assert list(df_pool.columns) == list(df_train.columns)
    assert df_pool.isna().sum().sum() == 0
    assert (df_pool[TARGET_COL].value_counts() == N_MAX).all()

    POOL_FILE = os.path.join(DATA_DIR, f"{STEM}_pool.csv")
    df_pool.to_csv(POOL_FILE, index=False)
    print("written:", POOL_FILE, f"({len(df_pool):,} rows)")
    if IN_COLAB:
        files.download(POOL_FILE)

In [ ]:
# Keep the checkpoint: it resumes an interrupted run and rebuilds the pool without retraining.
if IN_COLAB and not str(CHECKPOINT_DIR).startswith("/content/drive"):
    files.download(ckpt_path(STEM))